In [2]:
from pathlib import Path
import os, json, time
import duckdb
import numpy as np
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from sentence_transformers import SentenceTransformer

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

arts = con.execute(f"""
    SELECT article_id, prod_name, product_type_name, colour_group_name,
           graphical_appearance_name, index_group_name, department_name,
           COALESCE(detail_desc, '') AS detail_desc
    FROM '{(DATA / 'articles.parquet').as_posix()}' ORDER BY article_id
""").df()
emb = np.load(DATA / "article_emb.npy")
emb_ids = np.load(DATA / "article_emb_ids.npy")
assert (arts["article_id"].to_numpy() == emb_ids).all(), "Row order mismatch"

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

load_dotenv()
print("Keys found in .env:", list(dotenv_values().keys()))   # names only, never values
print(f"Articles: {len(arts):,} | embeddings: {emb.shape}")

Keys found in .env: ['GEMINI_API_KEY']
Articles: 105,542 | embeddings: (105542, 384)


In [3]:
available = con.execute(f"""
    SELECT DISTINCT article_id FROM '{(DATA / 'transactions_train.parquet').as_posix()}'
    WHERE t_dat > DATE '2020-09-22' - INTERVAL 56 DAY
""").df()["article_id"]
mask = arts["article_id"].isin(set(available)).to_numpy()
cat = arts[mask].reset_index(drop=True)
cat_emb = emb[mask]
cat["product_code"] = cat["article_id"].str[:7]
print(f"Searchable catalog: {len(cat):,} items currently selling")

def retrieve(query, k=10):
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = cat_emb @ q
    order = np.argsort(-scores)
    out, seen = [], set()
    for i in order:
        pc = cat.at[i, "product_code"]
        if pc in seen:
            continue
        seen.add(pc)
        r = cat.iloc[i]
        out.append({"article_id": r["article_id"], "name": r["prod_name"],
                    "type": r["product_type_name"], "colour": r["colour_group_name"],
                    "section": r["index_group_name"], "desc": r["detail_desc"][:200],
                    "score": round(float(scores[i]), 3)})
        if len(out) == k:
            break
    return out

for q in ["black dress for a summer wedding", "warm winter jacket for men", "kids pyjamas with animal print"]:
    print(f"\n>>> {q}")
    for r in retrieve(q, k=5):
        print(f"   {r['score']}  {r['name']} | {r['type']} | {r['colour']} | {r['section']}")

Searchable catalog: 35,576 items currently selling

>>> black dress for a summer wedding
   0.587  Summer nights dress | Dress | Black | Ladieswear
   0.583  Summer dress | Dress | Black | Ladieswear
   0.556  Best Day Wedding Top | Top | Off White | Ladieswear
   0.553  Summer strap dress | Dress | Black | Divided
   0.552  Bridesmaid | Dress | Light Blue | Ladieswear

>>> warm winter jacket for men
   0.558  Go light jacket | Jacket | Dark Green | Sport
   0.549  Snow short jacket | Jacket | Black | Baby/Children
   0.536  Andreas | Jacket | Black | Menswear
   0.533  Dragomir Jacket | Jacket | Black | Sport
   0.531  Rocket hooded jacket | Jacket | Dark Orange | Menswear

>>> kids pyjamas with animal print
   0.571  PETAL PJ | Pyjama set | White | Baby/Children
   0.567  Leon l/l pj SB | Pyjama set | Dark Blue | Baby/Children
   0.567  TILDA PJ | Pyjama set | Light Green | Baby/Children
   0.566  4PCS LS PJ Girly with Frill | Pyjama set | Light Red | Baby/Children
   0.559  Low Pric

In [4]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=60_000))   # 60 s per request
MODELS = ["gemini-flash-latest", "gemini-flash-lite-latest"]           # backup if the first is overloaded

SYSTEM = """You are a shopping assistant for H&M.
You may ONLY recommend products from the PRODUCTS list you are given.
Never invent products, names, colours, prices or details that are not in the list.
Pick up to 5 products that best match the request.
Leave out products that only partly match (wrong colour, wrong type, wrong section such as kids vs men).
If nothing in the list matches, say so honestly and recommend nothing.
Reply as JSON with two keys:
"answer": 2 to 4 short, friendly sentences for the customer, naming the products you picked.
"article_ids": the article_id values you recommend, copied exactly from the list."""

def call_gemini(prompt, retries=6):
    wait = 5
    for attempt in range(1, retries + 1):
        model_name = MODELS[(attempt - 1) % len(MODELS)]
        try:
            resp = client.models.generate_content(
                model=model_name, contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM,
                                                   response_mime_type="application/json",
                                                   temperature=0))
            return resp, model_name
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  attempt {attempt} with {model_name} failed ({type(e).__name__}). Waiting {wait}s...")
            time.sleep(wait)
            wait = min(wait * 2, 40)

def ask(query, k=20):
    items = retrieve(query, k)
    shown = [{key: r[key] for key in ["article_id", "name", "type", "colour", "section", "desc"]} for r in items]
    prompt = f"REQUEST: {query}\n\nPRODUCTS:\n{json.dumps(shown, ensure_ascii=False)}"
    resp, used = call_gemini(prompt)
    return items, json.loads(resp.text), used

t0 = time.time()
items, out, used = ask("warm winter jacket for men")
print(f"Model used: {used} | {time.time() - t0:.1f}s\n")
print(out["answer"], "\n")
names = {r["article_id"]: f"{r['name']} | {r['type']} | {r['colour']} | {r['section']}" for r in items}
for a in out["article_ids"]:
    print(" ", a, "->", names.get(a, "NOT IN RETRIEVED LIST"))

  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
Model used: gemini-flash-lite-latest | 26.1s

I found some great padded and down jackets for men that will help keep you warm this winter, including the Peter downfree jacket, the Uno lightweight down jacket, and the Mr Tom padded jacket. Take a look and stay cozy! 

  0778645001 -> Peter downfree jacket | Unknown | Black | Menswear
  0899020002 -> Uno | Jacket | Dark Blue | Menswear
  0790820001 -> Mr Tom padded jacket | Jacket | Black | Menswear


## AI product search (retrieval + Gemini)

This lets someone type what they want in plain words. It works in two steps. First I turn the request into a vector with the same sentence-transformer I used for products, and pull the 20 closest products from the 35,576 items that sold in the last 8 weeks. I keep only one colour per product, because otherwise the list fills up with the same item in different colours. Then I send the request and those 20 products to Gemini, with strict instructions to recommend only from that list, drop partial matches, and say so if nothing fits. It replies in JSON so I can check its answer automatically.

Retrieval on its own was good at the type of item but bad at hard rules. For "warm winter jacket for men", only 2 of the top 5 were actually men's jackets. The rest were kids' or sportswear. Gemini fixed that: from the 20 candidates it picked three men's jackets and nothing else.

Gemini's servers were overloaded during my first tries (error 503), and one request hung for over 3 minutes. So I added a 60-second timeout, automatic retries with growing waits, and a fallback to a lighter Gemini model. That's the version I'll use in the app.

In [5]:
print("Sections:", sorted(cat["index_group_name"].unique()))
print("\nColours:", sorted(cat["colour_group_name"].unique()))
print("\nTop 50 product types:")
print(cat["product_type_name"].value_counts().head(50).to_string())

Sections: ['Baby/Children', 'Divided', 'Ladieswear', 'Menswear', 'Sport']

Colours: ['Beige', 'Black', 'Blue', 'Bronze/Copper', 'Dark Beige', 'Dark Blue', 'Dark Green', 'Dark Grey', 'Dark Orange', 'Dark Pink', 'Dark Purple', 'Dark Red', 'Dark Turquoise', 'Dark Yellow', 'Gold', 'Green', 'Greenish Khaki', 'Grey', 'Greyish Beige', 'Light Beige', 'Light Blue', 'Light Green', 'Light Grey', 'Light Orange', 'Light Pink', 'Light Purple', 'Light Red', 'Light Turquoise', 'Light Yellow', 'Off White', 'Orange', 'Other', 'Other Blue', 'Other Green', 'Other Orange', 'Other Pink', 'Other Purple', 'Other Red', 'Other Turquoise', 'Other Yellow', 'Pink', 'Purple', 'Red', 'Silver', 'Transparent', 'Turquoise', 'Unknown', 'White', 'Yellow', 'Yellowish Brown']

Top 50 product types:
product_type_name
Dress                4167
Trousers             4147
Sweater              2407
T-shirt              2166
Top                  1541
Blouse               1476
Shorts               1386
Jacket               1218
Ve

In [6]:
WOMEN = {"Ladieswear", "Divided"}
BLUES = {"Blue", "Dark Blue", "Light Blue"}

TESTS = [
    {"q": "black dress for a summer wedding",  "type": {"Dress"}, "section": WOMEN, "colour": {"Black"}},
    {"q": "warm winter jacket for men",        "type": {"Jacket"}, "section": {"Menswear"}},
    {"q": "kids pyjamas",                      "type": {"Pyjama set", "Pyjama bottom"}, "section": {"Baby/Children"}},
    {"q": "women's high waist jeans",          "type": {"Trousers"}, "section": WOMEN},
    {"q": "men's white t-shirt",               "type": {"T-shirt"}, "section": {"Menswear"}, "colour": {"White"}},
    {"q": "bikini top for the beach",          "type": {"Bikini top"}},
    {"q": "running leggings for women",        "type": {"Leggings/Tights"}, "section": WOMEN | {"Sport"}},
    {"q": "men's hoodie",                      "type": {"Hoodie"}, "section": {"Menswear"}},
    {"q": "blue denim skirt",                  "type": {"Skirt"}, "colour": BLUES},
    {"q": "women's knitted sweater",           "type": {"Sweater"}, "section": WOMEN},
    {"q": "earrings",                          "type": {"Earring"}},
    {"q": "men's shorts for summer",           "type": {"Shorts"}, "section": {"Menswear"}},
    {"q": "heeled sandals for a party",        "type": {"Heeled sandals"}},
    {"q": "white blouse for the office",       "type": {"Blouse"}, "section": WOMEN, "colour": {"White"}},
    {"q": "laptop charger",                    "none": True},
    {"q": "garden lawn mower",                 "none": True},
]

# Safety check: every rule value must exist in the data
known = {"type": set(cat["product_type_name"]), "section": set(cat["index_group_name"]),
         "colour": set(cat["colour_group_name"])}
bad = [(t["q"], f, v) for t in TESTS for f in ("type", "section", "colour")
       for v in t.get(f, set()) if v not in known[f]]
print("Unknown rule values:", bad if bad else "none ✅")

def relevant(item, t):
    if t.get("none"):
        return False
    return (item["type"] in t["type"]
            and ("section" not in t or item["section"] in t["section"])
            and ("colour" not in t or item["colour"] in t["colour"]))

rows = []
for t in TESTS:
    if t.get("none"):
        continue
    top5 = retrieve(t["q"], k=5)
    rows.append({"query": t["q"], "retrieval_P@5": sum(relevant(r, t) for r in top5) / 5})

ret_df = pd.DataFrame(rows)
print(ret_df.to_string(index=False))
print(f"\nMean retrieval precision@5: {ret_df['retrieval_P@5'].mean():.2f}")

Unknown rule values: none ✅
                           query  retrieval_P@5
black dress for a summer wedding            0.6
      warm winter jacket for men            0.4
                    kids pyjamas            0.6
        women's high waist jeans            0.8
             men's white t-shirt            0.6
        bikini top for the beach            1.0
      running leggings for women            1.0
                    men's hoodie            0.8
                blue denim skirt            1.0
         women's knitted sweater            1.0
                        earrings            0.8
         men's shorts for summer            0.8
      heeled sandals for a party            0.8
     white blouse for the office            0.6

Mean retrieval precision@5: 0.77


In [7]:
results = []
for i, t in enumerate(TESTS, 1):
    items, out, used = ask(t["q"], k=20)
    ids = out.get("article_ids", [])
    by_id = {r["article_id"]: r for r in items}
    picks = [by_id[a] for a in ids if a in by_id]
    results.append({
        "query": t["q"],
        "model": used.replace("gemini-", ""),
        "picked": len(ids),
        "grounded": all(a in by_id for a in ids),
        "precision": (sum(relevant(r, t) for r in picks) / len(picks)) if (picks and not t.get("none")) else None,
        "relevant_in_20": sum(relevant(r, t) for r in items),
        "none_test": bool(t.get("none")),
        "answer": out.get("answer", ""),
    })
    print(f"{i:>2}/{len(TESTS)} done: {t['q']}")
    time.sleep(4)

ev = pd.DataFrame(results)
ev.to_json(DATA / "search_eval.json", orient="records", indent=2)   # saved, so no need to call Gemini again
print()
print(ev.drop(columns=["answer"]).to_string(index=False))

  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 1/16 done: black dress for a summer wedding
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 2/16 done: warm winter jacket for men
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 3/16 done: kids pyjamas
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 4/16 done: women's high waist jeans
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 5/16 done: men's white t-shirt
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 6/16 done: bikini top for the beach
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 7/16 done: running leggings for women
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 8/16 done: men's hoodie
  attempt 1 with gemini-flash-latest failed (ServerError). Waiting 5s...
 9/16 done: blue denim skirt
  attempt 1 with gemini-flash-lates

In [8]:
ev = pd.read_json(DATA / "search_eval.json")
real = ev[~ev["none_test"]].merge(ret_df, on="query")

print(real[["query", "retrieval_P@5", "precision", "picked", "relevant_in_20"]].to_string(index=False))

print(f"\nRetrieval only, mean precision@5: {real['retrieval_P@5'].mean():.2f}")
print(f"Gemini picks, mean precision:     {real['precision'].mean():.2f}  (over {real['precision'].notna().sum()} queries with picks)")
print(f"Grounded (all picks from list):   {ev['grounded'].sum()}/{len(ev)}")
none = ev[ev["none_test"]]
print(f"Nonsense requests refused:        {(none['picked'] == 0).sum()}/{len(none)}")
print(f"Queries where Gemini picked 0:    {(real['picked'] == 0).sum()}")
print("Models used:", ev["model"].value_counts().to_dict())

                           query  retrieval_P@5  precision  picked  relevant_in_20
black dress for a summer wedding            0.6        1.0       3               8
      warm winter jacket for men            0.4        0.5       2              11
                    kids pyjamas            0.6        1.0       3              10
        women's high waist jeans            0.8        1.0       4              14
             men's white t-shirt            0.6        1.0       3              11
        bikini top for the beach            1.0        1.0       4              20
      running leggings for women            1.0        1.0       4              18
                    men's hoodie            0.8        1.0       3              16
                blue denim skirt            1.0        1.0       5              20
         women's knitted sweater            1.0        1.0       3              14
                        earrings            0.8        0.8       5              19
    

## Evaluating the AI search

I tested the search on 16 requests. For 14 of them I wrote strict rules for what counts as a correct product, using the exact category names in the data (for example "men's hoodie" means type Hoodie and section Menswear). The other 2 were things H&M doesn't sell, a laptop charger and a lawn mower, to check that Gemini would admit there was nothing to recommend.

Retrieval on its own got a mean precision@5 of 0.77. It was good at the type of product but often ignored colour or section. Its worst case was "warm winter jacket for men" at 0.4.

Gemini's picks were right 95% of the time. Only two requests weren't perfect: the men's jacket (1 of 2 picks was wrong) and earrings (4 of 5 right). The two numbers aren't a fair head-to-head, though. Retrieval is scored on its top 5, while Gemini usually picked only 2 to 5 items, and picking fewer makes precision easier.

The result I care about most is grounding. In all 16 cases, every product Gemini recommended came from the list I gave it, and it recommended nothing for both nonsense requests. So it didn't invent any products.

Some limits. 16 requests is a small test, and I wrote the rules myself. The main Gemini model was overloaded during the whole run, so all 16 answers came from the lighter fallback model. I saved the results to a file so I could analyse them without calling the API again.